# Guardian AI — MuRIL Fine-Tuning Notebook

Fine-tunes `google/muril-base-cased` on the Guardian AI safety dataset.

**Requirements:**
- Runtime → T4 GPU (Runtime > Change runtime type > T4)
- Upload `train.csv` and `test.csv` in Step 2

**Output:** Fine-tuned model saved to Google Drive at `MyDrive/muril_finetuned/`

## Step 1 — Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate
print('Done installing.')

## Step 2 — Upload train.csv and test.csv
Run this cell, then click **Choose Files** and upload both files.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload train.csv and test.csv
print('Uploaded:', list(uploaded.keys()))

## Step 3 — Load & inspect data

In [ ]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f'Train rows : {len(train_df)}')
print(f'Test rows  : {len(test_df)}')
print()
print('Train label distribution:')
print(train_df['label'].value_counts().rename({0: 'SAFE', 1: 'EMERGENCY'}))
print()
print('Sample rows:')
train_df.sample(5)

## Step 4 — Tokenize

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'google/muril-base-cased'
MAX_LEN    = 128

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SafetyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['text'].tolist()
        self.labels    = df['label'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SafetyDataset(train_df, tokenizer, MAX_LEN)
eval_dataset  = SafetyDataset(test_df,  tokenizer, MAX_LEN)

print(f'Train dataset: {len(train_dataset)} samples')
print(f'Eval  dataset: {len(eval_dataset)} samples')
print('Tokenization complete.')

## Step 5 — Load MuRIL model

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'SAFE', 1: 'EMERGENCY'},
    label2id={'SAFE': 0, 'EMERGENCY': 1},
)
model.to(device)
print('Model loaded and moved to', device)

## Step 6 — Fine-tune with HuggingFace Trainer

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy':         accuracy_score(labels, preds),
        'f1_emergency':     f1_score(labels, preds, pos_label=1),
        'recall_emergency': recall_score(labels, preds, pos_label=1),
    }

training_args = TrainingArguments(
    output_dir             = './muril_finetuned',
    num_train_epochs       = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps           = 100,
    weight_decay           = 0.01,
    learning_rate          = 2e-5,
    eval_strategy          = 'epoch',
    save_strategy          = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'recall_emergency',  # prioritise catching emergencies
    logging_steps          = 20,
    report_to             = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    compute_metrics = compute_metrics,
)

print('Starting fine-tuning ...')
print('This takes ~10-20 minutes on a T4 GPU.')
print()
trainer.train()

## Step 7 — Final evaluation on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

results = trainer.evaluate()
print('\n=== Final Eval Results ===')
for k, v in results.items():
    print(f'  {k:<35} {v:.4f}')

# Detailed report
preds_output = trainer.predict(eval_dataset)
preds  = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print()
print('=== Classification Report ===')
print(classification_report(labels, preds, target_names=['SAFE', 'EMERGENCY']))

cm = confusion_matrix(labels, preds)
print('=== Confusion Matrix (rows=actual, cols=predicted) ===')
print(f'               Pred SAFE   Pred EMERGENCY')
print(f'  Actual SAFE      {cm[0][0]:>5}          {cm[0][1]:>5}')
print(f'  Actual EMER      {cm[1][0]:>5}          {cm[1][1]:>5}')

## Step 8 — Save model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/muril_finetuned'

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print()
print(f'Model saved to Google Drive: {SAVE_PATH}')
print()
print('Next steps:')
print('  1. Download the muril_finetuned/ folder from Drive')
print('  2. Put it in your GuardianAi/ project folder')
print('  3. Run: python evaluate.py --model ./muril_finetuned')
print('     to compare with the baseline score.')